<a href="https://colab.research.google.com/github/intern1-crypto/ETL-colab/blob/main/ETL%E9%81%8B%E7%94%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#【注意⚠️】
変更を施したらGitHubにプッシュしてください！
ファイル＞GitHubにコピーを保存

※変更内容をコミットメッセージに記入すること


#【使い方】
**ランタイム＞すべてのセルを実行**を押してください！

<p>「実行完了しました」というポップアップが表示されたらOKを押して終了(全体で150~200秒)<p>

-----

<p>途中でノートブックのGoogle認証情報とドライブへのアクセスの許可を求められるので許可する。<p>

ドライブへのアクセス

- このノートブックにGoogleドライブのファイルへのアクセスを許可しますか？
   <br>→ **Google Driveに接続**
-
アカウントを選択してください
   <br>→ **(選択)**
- Google Drive for Desktop にログイン
   <br>→ **次へ**
- Google Drive for Desktop がGoogleアカウントへのアクセスを求めています
   <br>→ **続行**


Google認証情報へのアクセス

- Google認証情報へのアクセスをこのノートブックに許可しますか？
   <br>→ **許可**
- アカウントを選択してください
   <br>→ **(選択)**
- Third-party authored notebook code にログイン
   <br>→ **次へ**
- Third-party authored notebook code がGoogleアカウントへのアクセスを求めています
   <br>→ **続行**

[マニュアル](https://docs.google.com/document/d/1T7Q7Vsnb1YEY-Vo7W1PcUNEfkuv8J7xQcccuYaGCxo0/edit?tab=t.6iyed43s2oim)

# 初期設定

In [ ]:
# ライブラリインポート
import numpy as np
import pandas as pd
import glob
from gspread_dataframe import get_as_dataframe, set_with_dataframe

In [ ]:
# Google Driveマウント(お約束その1)
from google.colab import drive
drive.mount('/content/drive')

# 各データがまとめられているフォルダのパス
path = "/content/drive/MyDrive/過去データ"

Mounted at /content/drive


In [ ]:
# Google認証(お約束その2)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
# 店舗名と対応する店舗番号の辞書
store_dict = {
    "同志社大学前店": 101,
    "京都大学前店": 102,
    "早稲田大学前店": 103,
    "名古屋大学前店": 104,
    "東京大学(本郷)前店": 105,
    "神戸大学前店": 106,
    "慶應義塾大学前店": 107,
    "関西大学前店": 108,
    "大阪大学前店": 110,
    "立命館大学(衣笠)前店": 111,
    "九州大学前店": 112,
    "立命館大学(びわこ・くさつ)店": 114,
    "東京科学大学前店": 115,
    "一橋大学前店": 116,
    "みなと銀行学園都市店": 117,
    "関西学院大学(上ヶ原)前店": 118,
    "東京大学(駒場)前店": 119,
    "みなと銀行武庫川女子大学店": 120,
    "東京理科大学(神楽坂)店": 121,
    "BiZCAFE(関西学院大学神戸三田)店": 422,
    "BiZCAFE（千葉大学）店": 423,
    "岡山大学前店": 124,
    "北九州市立大学店": 125,
    "広島大学店": 126,
    "横浜国立大学店": 127,
    "IIT DELHI": 901,
    "IIT ROORKEE": 902,
    "IIT BOMBAY": 903,
    "IIT HYDERABAD": 904,
    "オンラインストア(立命館APU)店": 801,
    "検証用": 999
}

# store_dictのキーと値を逆にした辞書
inverse_store_dict = {v: k for k, v in store_dict.items()}

# 参加データ

## データ抽出

In [ ]:
# Google Drive内のCSVファイルパスを指定
csv_files_meetup = glob.glob(f'{path}/人別_Meetup参加者_2510更新/*.csv')

# CSVファイルを読み込む
df_list_meetup = []
df_list_meetup += [pd.read_csv(file, encoding='cp932') for file in csv_files_meetup]

# データフレームを縦に結合
df1 = pd.concat(df_list_meetup, ignore_index=True)

df1.columns

Index(['結合ID', '予約ID', '会員ID', '店舗名', '大学', '学部', '学年', '文理', '性別', '専攻',
       '出身地', '出身地での就職希望', '卒業年月', '入学年月', '予約日', 'イベント日', 'イベント時間', '開催形式',
       '交流会テーマ', '企業名', '開催店舗', '満足度', '開催企業への要望', '開催企業へのメッセージ', '参加可否',
       'キャンセル有無', '予約したきっかけ', '更新日時', '店舗番号'],
      dtype='object')

In [ ]:
df1['イベント日'] = pd.to_datetime(df1['イベント日'])

In [ ]:
## スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/1B33R0VydVlKLH7UrDwyAWZEwEksFrhgDSpGZbMktDKQ/edit?gid=0#gid=0"
ss_meetup = gc.open_by_url(url)
print(f'{ss_meetup.title}を開きました')

# スプレッドシートを読み込みデータフレーム化
st_meetup = ss_meetup.get_worksheet(0)
df_ss_meetup = get_as_dataframe(st_meetup)
df_ss_meetup.shape

# スプレッドシートを読み込みデータフレーム化
st_meetup_del = ss_meetup.worksheet("カウント対象外Meetup")
df_ss_meetup_del = get_as_dataframe(st_meetup_del)
df_ss_meetup_del.shape

raw_admin_Meetup_人別を開きました


(2, 7)

In [ ]:
df_ss_meetup_del.columns

Index(['MeetupID', '企業名', '開催日', '開催時間', '参加数', '予約数', 'キャンセル'], dtype='object')

In [ ]:
# 削除処理
len_before1 = len(df_ss_meetup)
df_ss_meetup = df_ss_meetup[~df_ss_meetup['予約ID'].isin(df_ss_meetup_del['MeetupID'])]
len_after1 = len(df_ss_meetup)
print(f'{len_before1 - len_after1}行の重複データが削除されました')

2行の重複データが削除されました


In [ ]:
df_ss_meetup.columns

Index(['結合ID', '予約ID', '会員ID', '店舗名', '大学', '学部', '学年', '文理', '性別', '専攻',
       '出身地', '出身地での就職希望', '卒業年月', '入学年月', '予約日', 'イベント日', 'イベント時間', '開催形式',
       '交流会テーマ', '企業名', '開催店舗', '満足度', '開催企業への要望', '開催企業へのメッセージ', '参加可否',
       'キャンセル有無', '予約したきっかけ', '部活ID', '参加経由', '注力可否', '注力開始日時', '新更新日時',
       '={"店舗番号";ARRAYFORMULA(iferror(vlookup(D2:D,'店舗番号'!$A$2:$B$30,2,false)))}'],
      dtype='object')

In [ ]:
# 日時を変換
df_ss_meetup['イベント日'] = pd.to_datetime(df_ss_meetup['イベント日'], format='mixed')

# yyyy-MM-dd HH:mm:ss 形式に変換
df_ss_meetup['イベント日'] = df_ss_meetup['イベント日'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
df_list = df_list_meetup + [df_ss_meetup]
file_names = [csv_files_meetup[i].split('/')[-1] for i in range(len(df_list) - 1)] + ["SpreadSheet"]

# ファイル情報
file_inf = [
    [file_names[i],
     len(df_list[i]),
     len(df_list[i].columns),
     pd.to_datetime(df_list[i]['イベント日'], errors='coerce').min(),
     pd.to_datetime(df_list[i]['イベント日'], errors='coerce').max()]
    for i in range(len(df_list))
]

df_file_inf = pd.DataFrame(file_inf, columns=['ファイル名', 'レコード数', 'カラム数', 'min', 'max'])
df_file_inf

,ファイル名,レコード数,カラム数,min,max
0,output_20231001-20240930.csv,11732,28,2023-10-03,2024-09-30
1,output_20241001-20250930.csv,17145,29,2024-10-01,2025-09-30
2,SpreadSheet,41840,33,2024-10-01,2026-07-31


In [ ]:
# 外部結合
df1 = pd.concat([df1, df_ss_meetup], join='outer', ignore_index=True)
df1.shape

(70717, 35)

In [ ]:
# 重複データ削除
len_before = len(df1)
df1.drop_duplicates(inplace=True)
len_after = len(df1)
print(f'{len_before - len_after}行の重複データが削除されました')

1行の重複データが削除されました


## データ処理

In [ ]:
df2 = df1.copy()

In [ ]:
# イベント時間から開始時刻を抽出
df2['開始時刻'] = df2['イベント時間'].str.extract(r'(\d{2}:\d{2}:\d{2})')

# カラム作成
df2['イベント開始日時'] = pd.to_datetime(df2['イベント日'].astype(str) + ' ' + df2['開始時刻'])
df2['イベント終了日時'] = df2['イベント開始日時'] + pd.Timedelta(hours=1)

# カラムを消す
df2.drop(columns=['開始時刻','イベント時間'], inplace = True)

# 日付型に
df2['イベント日'] = pd.to_datetime(df2['イベント日']).dt.date
df2.columns

/tmp/ipykernel_4062/1613317616.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2['イベント開始日時'] = pd.to_datetime(df2['イベント日'].astype(str) + ' ' + df2['開始時刻'])


Index(['結合ID', '予約ID', '会員ID', '店舗名', '大学', '学部', '学年', '文理', '性別', '専攻',
       '出身地', '出身地での就職希望', '卒業年月', '入学年月', '予約日', 'イベント日', '開催形式', '交流会テーマ',
       '企業名', '開催店舗', '満足度', '開催企業への要望', '開催企業へのメッセージ', '参加可否', 'キャンセル有無',
       '予約したきっかけ', '更新日時', '店舗番号', '部活ID', '参加経由', '注力可否', '注力開始日時', '新更新日時',
       '={"店舗番号";ARRAYFORMULA(iferror(vlookup(D2:D,'店舗番号'!$A$2:$B$30,2,false)))}',
       'イベント開始日時', 'イベント終了日時'],
      dtype='object')

In [ ]:
# 店舗名をもとに店舗番号をマッピング
df2['店舗番号'] = df2['店舗名'].map(store_dict)

if(df2['店舗名'].count() == df2['店舗番号'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(df2[df2['店舗番号'].isnull()]['店舗名'].unique())

マッピング成功


In [ ]:
df2 = df2.rename(columns={'予約ID': 'MeetupID'})

In [ ]:
# 整数型にする
df2['MeetupID'] = df2['MeetupID'].astype(int)
df2['会員ID'] = df2['会員ID'].astype(int)

In [ ]:
# 参加：参加なら1, else 0
df2['参加可否'] = (df2['参加可否'] == '参加').astype(int)
df2 = df2.rename(columns = {'参加可否': '参加'})

In [ ]:
# 参加予定: 参加してないかつキャンセル有無が欠損→1,else→0
df2['参加予定'] = ((df2['参加'] == 0) & (df2['キャンセル有無'].isnull())).astype(int)

In [ ]:
# 開催形式を簡潔に言い換え
df2['開催形式'] = df2['開催形式'].replace({
    'オンラインMeetup': 'オンライン',
    'Meetup (店舗開催)': '対面'
})

In [ ]:
df2['予約ID'] = df2['イベント開始日時'].astype(str) + "_" + df2['MeetupID'].astype(str) + '_' + df2['結合ID'].astype(str)

In [ ]:
# カラムを絞りdf5を作成
df5 = df2[['予約ID', 'イベント開始日時','イベント終了日時', 'MeetupID', '結合ID', '会員ID', '店舗番号', '店舗名',
     '企業名', '開催形式','参加','参加予定', 'キャンセル有無', '予約したきっかけ',
     '学年','入学年月','予約日','卒業年月', '大学', '文理', '学部', '専攻',
     '性別', '出身地', '満足度', '参加経由', '注力可否', '部活ID']]

In [ ]:
column_mapping = {
    '予約ID': 'reservation_id',
    'イベント開始日時': 'start_at',
    'イベント終了日時': 'end_at',
    'MeetupID': 'event_id',
    '結合ID': 'conected_id',
    '会員ID': 'member_id',
    '店舗番号': 'store_code',
    '店舗名': 'store',
    '企業名': 'company',
    '開催形式': 'meetup_type',
    '参加': 'attendance',
    '参加予定': 'planned_attendance',
    'キャンセル有無': 'cancell',
    '予約したきっかけ': 'reservation_reason',
    '学年': 'grade',
    '入学年月': 'enrollment_year_month',
    '予約日': 'reservation_day',
    '卒業年月': 'graduation_year_month',
    '大学': 'university',
    '文理': 'bunri',
    '学部': 'faculty',
    '専攻': 'major',
    '性別': 'gender',
    '出身地': 'hometown',
    '満足度': 'satisfaction',
    '参加経由': 'reservation_way',
    '注力可否': 'Pickup1',
    '部活ID': 'bukatsu_id'
}

# カラム名を英語に変換
df5.rename(columns=column_mapping, inplace=True)

/tmp/ipykernel_4062/3192320439.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5.rename(columns=column_mapping, inplace=True)


In [ ]:
#Pickup count
# 条件
cond_1 = ((df5["attendance"] == 1) | (df5["planned_attendance"] == 1)) & (df5["Pickup1"] == "注力している")
cond_2 = ((df5["attendance"] == 1) | (df5["planned_attendance"] == 1)) & (
    (df5["Pickup1"].isna()) | (df5["Pickup1"] == "注力していない")
)
cond_3 = (df5["attendance"] == 0) & (df5["planned_attendance"] == 0)

# 値の割り当て
df5["Pickup"] = np.select([cond_1, cond_2, cond_3], [2, 1, 0], default=np.nan)

#PickupをInt型に変換
df5["Pickup"] = df5["Pickup"].fillna(0).astype(int)

/tmp/ipykernel_4062/2921400732.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5["Pickup"] = np.select([cond_1, cond_2, cond_3], [2, 1, 0], default=np.nan)
/tmp/ipykernel_4062/2921400732.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5["Pickup"] = df5["Pickup"].fillna(0).astype(int)


In [ ]:
#Pickup1カラム削除
df5 = df5.drop(columns=["Pickup1"])

In [ ]:
bq_meetup = df5.copy()

# 来店データ

## データ抽出

In [ ]:
# Google Drive内のCSVファイルパスを指定
csv_files_order = glob.glob(f'{path}/来店数/*.csv')

# CSVファイルを読み込む
df_list_order = []
# Try reading with utf-8 encoding
try:
    df_list_order += [pd.read_csv(file, encoding='utf-8') for file in csv_files_order]
except UnicodeDecodeError:
    # If utf-8 fails, fall back to cp932
    df_list_order += [pd.read_csv(file, encoding='cp932') for file in csv_files_order]


# データフレームを縦に結合
df3 = pd.concat(df_list_order, ignore_index=True)

df3.shape

(809966, 14)

In [ ]:
## スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/1584pXhsuZfu-hmx9N1p9_JQ3V33uy-VFGdHkbfWxCxU/edit?gid=2012077172#gid=2012077172"
ss_order = gc.open_by_url(url)
print(f'{ss_order.title}を開きました')

raw_admin_来店を開きました


In [ ]:
# スプレッドシートデータ抽出
st_order = ss_order.get_worksheet(0)
print("データ抽出中...", end='', flush=True)
df_ss_order = get_as_dataframe(st_order)
print("\rデータ抽出完了", end='', flush=True)

データ抽出完了

In [ ]:
# yyyy-MM-dd HH:mm:ss 形式に変換
df_ss_order['登録日'] = pd.to_datetime(df_ss_order['登録日'], format='mixed')
df_ss_order['登録日'] = df_ss_order['登録日'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
df_list = df_list_order + [df_ss_order]
file_names = [csv_files_order[i].split('/')[-1] for i in range(len(df_list) - 1)] + ["SpreadSheet"]

# ファイル情報
file_inf = [
    [file_names[i],
     len(df_list[i]),
     len(df_list[i].columns),
     pd.to_datetime(df_list[i]['登録日'], errors='coerce').min(), # Convert to datetime here
     pd.to_datetime(df_list[i]['登録日'], errors='coerce').max()]  # Convert to datetime here
    for i in range(len(df_list))
]

df_file_inf = pd.DataFrame(file_inf, columns=['ファイル名', 'レコード数', 'カラム数', 'min', 'max'])
df_file_inf

,ファイル名,レコード数,カラム数,min,max
0,order_report_2023-10-01 ~ 2023-11-30.csv,87029,13,2023-10-02 10:48:51,2023-11-30 21:27:37
1,order_report_2023-12-01 ~ 2024-01-31.csv,65005,13,2023-12-01 10:48:04,2024-01-31 21:23:48
2,order_report_2024-02-01 ~ 2024-03-31.csv,64252,13,2024-02-01 12:31:01,2024-04-02 19:29:02
3,order_report_2024-04.csv,60241,13,2024-04-01 11:57:51,2024-04-30 21:28:13
4,order_report_2024-05.csv,60109,13,2024-05-01 10:12:11,2024-05-31 21:30:24
5,order_report_2024-06.csv,54924,13,2024-06-01 09:25:06,2024-06-30 15:46:25
6,order_report_2024-07.csv,63648,13,2024-07-01 10:47:48,2024-07-31 21:29:17
7,order_report_2024-08.csv,39066,13,2024-08-01 12:02:52,2024-08-30 20:58:51
8,order_report_2024-09.csv,38866,13,2024-09-02 12:02:55,2024-10-08 12:16:17
9,order_20241001-2025-0930.csv,276826,9,2024-10-01 10:46:00,2025-09-30 18:48:00


In [ ]:
# CSVデータとスプレッドシートデータの結合
df3 = pd.concat([df3, df_ss_order], join='outer', ignore_index=True)
df3.shape

(1080177, 16)

In [ ]:
# 重複データ削除
len_before = len(df3)
df3.drop_duplicates(subset=["結合ID", "オーダーID", "登録日"], inplace=True)
#df3.drop_duplicates(inplace=True)
len_after = len(df3)
print(f'{len_before - len_after}行の重複データが削除されました')

12行の重複データが削除されました


In [ ]:
df3.columns

Index(['結合ID', 'オーダーID', '店舗名', 'メールアドレス', 'グループ', '大学/キャンパス', '杯数', 'ドリンク',
       '学年', '学部・学科', 'タイプ', '金額', '登録日', '卒年',
       '={"店舗番号";ARRAYFORMULA(iferror(vlookup(C2:C,'店舗設定'!$A$1:$B$30,2,false)))}',
       '={"時間";ARRAYFORMULA(TIME(HOUR(I2:I), MINUTE(I2:I), SECOND(I2:I)))}'],
      dtype='object')

## データ処理

In [ ]:
df4 = df3.copy()

In [ ]:
df4 = df4[['結合ID', 'オーダーID', '店舗名', 'グループ', '大学/キャンパス', '学年', '卒年','学部・学科', 'タイプ',
       '登録日', 'ドリンク']]
dropped_columns = set(df3.columns) - set(df4.columns)
print(f'削除されたカラム: {dropped_columns}')
print(f'df4のカラム: {set(df4.columns)}')

削除されたカラム: {'メールアドレス', '={"時間";ARRAYFORMULA(TIME(HOUR(I2:I), MINUTE(I2:I), SECOND(I2:I)))}', '={"店舗番号";ARRAYFORMULA(iferror(vlookup(C2:C,\'店舗設定\'!$A$1:$B$30,2,false)))}', '杯数', '金額'}
df4のカラム: {'結合ID', 'タイプ', '大学/キャンパス', '学年', '登録日', '学部・学科', '店舗名', 'ドリンク', '卒年', 'オーダーID', 'グループ'}


In [ ]:
# 重複データ削除
len_before = len(df4)
df4.drop_duplicates(inplace=True)
len_after = len(df4)
print(f'{len_before - len_after}行の重複データが削除されました')

0行の重複データが削除されました


In [ ]:
# カラム名変更
df4 = df4.rename(columns={'登録日': 'オーダー日時'})

# オーダー日時をdatetime型に変換
df4['オーダー日時'] = pd.to_datetime(df4['オーダー日時'], format='mixed', dayfirst=False)

In [ ]:
# 店舗名をもとに店舗番号をマッピング
df4['店舗番号'] = df4['店舗名'].map(store_dict)

if(df4['店舗名'].count() == df4['店舗番号'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(df2[df2['店舗番号'].isnull()]['店舗名'].unique())

マッピング成功


In [ ]:

# オーダーIDを整数に
df4['オーダーID'] = pd.to_numeric(df4['オーダーID']).astype(int)

In [ ]:
df4['DU_id'] = df4['オーダー日時'].dt.date.astype(str) + '_' + df4['結合ID'] + "_" + df4['店舗番号'].astype(str)

In [ ]:
df4.columns

Index(['結合ID', 'オーダーID', '店舗名', 'グループ', '大学/キャンパス', '学年', '卒年', '学部・学科', 'タイプ',
       'オーダー日時', 'ドリンク', '店舗番号', 'DU_id'],
      dtype='object')

In [ ]:
# カラム名を変更
new_columns = ["conected_id", "order_id", "store", "group", "university", "grade","Graduation_year", "faculty", "type" , "ordered_at","drink", "store_code","DU_id"]
df4.columns = new_columns

In [ ]:
# 来店回数のカラム追加
df4["visit_count"] = df4.groupby("conected_id").cumcount() + 1

In [ ]:
# カラムの順番を入れ替え
df4 = df4[['order_id', 'ordered_at', 'store_code', 'store', 'conected_id', 'visit_count', 'group', 'grade', "Graduation_year",'university', 'faculty', 'type', 'DU_id', 'drink']]

In [ ]:
# 大学名整形
df4['university'] = df4['university'].str.replace(' /', '', regex=False)
df4['university'] = df4['university'].str.replace('大学 ', '大学', regex=False)
df4['university'] = df4['university'].str.replace('その他 ', 'その他', regex=False)

In [ ]:
# BigQueryに反映するデータフレームbq_order
# インド排除
bq_order = df4[df4['store_code'] < 500]

In [ ]:
bq_order['ordered_at'].max()

Timestamp('2026-06-29 20:20:00')

# イベントデータ

## データ抽出

In [ ]:
# Google Drive内のCSVファイルパスを指定
csv_files_event = glob.glob(f'{path}/枠別_Meetup開催企業_2510更新/*.csv')

# CSVファイルを読み込む
df_list_event = []
df_list_event += [pd.read_csv(file, encoding='cp932') for file in csv_files_event]

# データフレームを縦に結合
df6 = pd.concat(df_list_event, ignore_index=True)

df6.shape

(15098, 26)

In [ ]:
# 日付型に変換
df6['予約日'] = pd.to_datetime(df6['予約日'], format='mixed').dt.date

# 日時型に変換
df6['作成日'] = pd.to_datetime(df6['作成日'], format='mixed')
df6['キャンセル日'] = pd.to_datetime(df6['キャンセル日'], format='mixed')

In [ ]:
# スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/1bNd_oqfDEIvVysQSVGMdgWOxY39hSRZLgUBkVsRjysQ/edit?gid=0#gid=0"
ss_order = gc.open_by_url(url)
print(f'{ss_order.title}を開きました')

# スプレッドシートデータ抽出
st_event = ss_order.get_worksheet(0)
print("データ抽出中...", end='', flush=True)
df_ss_event = get_as_dataframe(st_event)
print("\rデータ抽出完了", end='', flush=True)

raw_admin_Meetup_枠別を開きました
データ抽出完了

In [ ]:
# 日付型に変換
df_ss_event['予約日'] = pd.to_datetime(df_ss_event['予約日'], format='mixed').dt.date

# 日時型に変換
df_ss_event['作成日'] = pd.to_datetime(df_ss_event['作成日'], format='mixed')
df_ss_event['キャンセル日'] = pd.to_datetime(df_ss_event['キャンセル日'], format='mixed')

In [ ]:
len_before = len(df_ss_event)

 # 予約IDが欠損しているレコードを削除
df_ss_event = df_ss_event.dropna(subset=['予約ID'])

len_after = len(df_ss_event)
print(f'{len_before - len_after}行のデータが削除されました')

0行のデータが削除されました


In [ ]:
df_list = df_list_event + [df_ss_event]
file_names = [csv_files_event[i].split('/')[-1] for i in range(len(df_list) - 1)] + ["SpreadSheet"]

# ファイル情報
file_inf = [
    [file_names[i],
     len(df_list[i]),
     len(df_list[i].columns),
     df_list[i]['予約日'].min(),
     df_list[i]['予約日'].max()]
    for i in range(len(df_list))
]

df_file_inf = pd.DataFrame(file_inf, columns=['ファイル名', 'レコード数', 'カラム数', 'min', 'max'])
df_file_inf

TypeError: '<=' not supported between instances of 'datetime.date' and 'float'

In [ ]:
# CSVデータとスプレッドシートデータの結合
df6 = pd.concat([df6, df_ss_event], join='outer', ignore_index=True)
df6 = df6.drop_duplicates()

## データ処理

In [ ]:
df7 = df6.copy()

In [ ]:
# 開始日時と終了日時を作成
df7["開始日時"] = pd.to_datetime(df7["予約日"].astype(str) + " " + df7["開始時間"])
df7["終了日時"] = pd.to_datetime(df7["予約日"].astype(str) + " " + df7["終了時間"])



df7.drop(columns=['開始時間', '終了時間', '予約日'], inplace = True)

In [ ]:
df7.columns

In [ ]:
# 整数型に
df7 = df7.astype({'予約ID': int, '企業様人数': int, '学生参加枠': int, '学生参加人数': int})

In [ ]:
# カラム絞る
df7 = df7[['予約ID', 'お客様企業名', '店舗名', '開催状態', '予約形態', '企業様人数', '学生参加枠',
       '対象卒年(Meetup)', 'タイプ(Meetup)', '学生予約人数', '学生参加人数', '作成日',
       'キャンセル日','注力可否', '開始日時', '終了日時']]

In [ ]:
df7['タイプ(Meetup)'].value_counts()

In [ ]:
df7 = df7.replace({"タイプ(Meetup)": {"オンラインMeetup (自宅から参加)": "オンライン", "Meetup (店舗開催)": "対面"}})

In [ ]:
df7.rename(columns={'予約ID': 'イベントID'}, inplace=True)

In [ ]:
# カラム名の変更
df7 = df7.rename(columns={
    'イベントID': 'event_id',
    'お客様企業名': 'company',
    '店舗名': 'store',
    '開催状態': 'status',
    '予約形態': 'event_type',
    '企業様人数': 'company_attendance',
    '学生参加枠': 'student_attendance_max',
    '対象卒年(Meetup)': 'target',
    'タイプ(Meetup)': 'meetup_type',
    '学生予約人数': 'student_reserved',
    '学生参加人数': 'student_attendance',
    '作成日': 'created_at',
    'キャンセル日': 'canceled_at',
    '注力可否': 'pickup',
    '開始日時': 'start_at',
    '終了日時': 'end_at'
})

In [ ]:
bq_event = df7.copy()

In [ ]:
df7['event_type'].value_counts()

# 日報データ

## データ抽出

APIサービスアカウントの作り方

In [ ]:
from googleapiclient.discovery import build
from google.oauth2 import service_account

# フォームの回答とフォームの構造を読み取るアクセス権限
SCOPES = [
    "https://www.googleapis.com/auth/forms.responses.readonly",
    "https://www.googleapis.com/auth/forms.body.readonly"
]

# 秘密鍵が含まれるJSON
SERVICE_ACCOUNT_FILE = f'{path}/daily-report-451612-0a0691c91c6b.json'

# Google APIの認証
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

# APIを操作するオブジェクトの作成(?)
service = build("forms", "v1", credentials=credentials)

In [ ]:
responses = []
page_token = None

# フォームIDを指定（フォームの編集URLで/d/と/editに挟まれる文字列）
form_id = "16jvJ4z6reUwimTsVZQsU6s0Rf3US2VpQpA7k7amNOr0"

# 一度に最大5000件しか取得できないので全て取得するまで回す
while True:
    # API リクエスト
    response = service.forms().responses().list(
        formId=form_id,
        pageSize=5000,
        pageToken=page_token
    ).execute()

    # データを追加
    responses.extend(response.get("responses", []))

    # 次のページがあるか確認
    page_token = response.get("nextPageToken")
    if not page_token:
        break  # 次のページがなければ終了

In [ ]:
# 各回答を処理して最初に質問IDを使ってDataFrameを作成
data = []
for res in responses:
    row = {
        "responseId": res["responseId"],
        "timestamp": res["createTime"],
    }
    for question_id, answer in res.get("answers", {}).items():
        # 質問IDごとの回答を追加
        row[question_id] = answer.get("textAnswers", {}).get("answers", [{}])[0].get("value", "")
    data.append(row)

# 最初に質問IDを使ってDataFrameを作成
df_report_raw = pd.DataFrame(data)

df_report_raw.shape

(53938, 511)

In [ ]:
df_report_raw.shape

(53938, 511)

In [ ]:
# フォームの構造を取得
form = service.forms().get(formId=form_id).execute()

In [ ]:
import re

def add_section_to_items(items):
    current_section = 0

    for item in items:
        # itemが辞書であることを確認
        if not isinstance(item, dict):
            continue

        # 1. ページ区切りの判定と更新
        if 'pageBreakItem' in item:
            title = item.get('title', '')
            # タイトルから数字部分を抽出
            match = re.search(r'(\d+)', title)
            if match:
                current_section = int(match.group(1))
            else:
                current_section = 0

        # 2. 質問アイテムへのsection追加
        elif 'questionItem' in item:
            item['section'] = current_section

    return items

# --- 実行 ---
form_stracture = add_section_to_items(form.get('items', []))

In [ ]:
# 質問IDと質問を対応させるマッピングを作成(全店共通セクションのみ)
question_mapping_common = {}
for item in form_stracture:
  store_code = item.get("section", {})
  if store_code != 0:
    continue
  question_id = item.get("questionItem", {}).get("question", {}).get("questionId", "No ID")
  question_title = item.get("title", {})
  question_mapping_common[question_id] = question_title

In [ ]:
question_mapping_common

{'4d501cce': 'スタッフ名（フルネーム）',
 '4b24acca': 'スタッフナンバー（7桁）',
 '1f88b1ab': '店舗名（シフトインした店舗）',
 '28408ab4': 'シフトイン日時',
 '683e4714': '接客数',
 '4a8a3472': '対象数',
 '0c9990b7': '未告知数',
 '265c7e57': '告知までの課題',
 '3831f946': '誘致で落ちた落とし穴',
 '1a1ae280': '誘致数',
 '6439d52a': '誘致した企業・卒年・人数',
 '239f00e8': '誘致のTips',
 '0ad9e46e': 'サービス連携',
 '73ee63a3': 'SHIRURU在庫状況\nシフトを上がったタイミングで「在庫0かどうか」を選択してください。',
 '7400e346': 'どんな言葉が刺さった？',
 '271c095d': '価値提供できなかった人は何が足りなかったのか振り返ろう！\n（全員に価値提供できていたら不要）',
 '637a8ed3': '企業を選択してください（複数選択可）'}

In [ ]:
# カラム名を質問内容に変換
df_report_raw.rename(columns=question_mapping_common, inplace=True)

In [ ]:
df_report_raw.columns

Index(['responseId', 'timestamp', '店舗名（シフトインした店舗）', '33044c18', '対象数',
       'スタッフ名（フルネーム）', '告知までの課題',
       'SHIRURU在庫状況\nシフトを上がったタイミングで「在庫0かどうか」を選択してください。', 'スタッフナンバー（7桁）',
       'シフトイン日時',
       ...
       '70de9e5b', '13be3450', '28d64205', '60816d0b', '7c1cbc62', '112a9c42',
       '30cbf9e9', '4e128986', '13936530', '1d54b979'],
      dtype='object', length=511)

In [ ]:
df_report_raw.shape

(53938, 511)

## データ処理

In [ ]:
df9 = df_report_raw.copy()

In [ ]:
df9.shape

(53938, 511)

In [ ]:
store_dict_report = {
    '111立命館大学(衣笠)前店': 111,
    '108関西大学前店': 108,
    '105東京大学(本郷)店': 105,
    '101同志社大学前店': 101,
    '116一橋大学前店': 116,
    '106神戸大学前店': 106,
    '110大阪大学前店': 110,
    '422BiZCAFE関西学院大学(神戸三田)店': 422,
    '115東京工業大学前店': 115,
    '105東京大学(本郷)前店': 105,
    '103早稲田大学前店': 103,
    '119東京大学(駒場)前店': 119,
    '423BiZCAFE千葉大学店': 423,
    '118関西学院大学(上ヶ原)前店': 118,
    '107慶應義塾大学前店': 107,
    '104名古屋大学前店': 104,
    '117みなと銀行学園都市店': 117,
    '112九州大学前店': 112,
    '121東京理科大学(神楽坂)店': 121,
    '120みなと銀行武庫川女子店': 120,
    '102京都大学前店': 102,
    '114立命館大学(びわこ・くさつ)店': 114,
    '京都大学前店': 102,
    '名古屋': 104,
    '104名古屋前店': 104,
    '423BiZCAFE千葉大学前店': 423,
    '107慶応義塾大学前店': 107,
    '124岡山大学前店': 124,
    '125北九州市立大学店': 125,
    '126広島大学店': 126,
    '127横浜国立大学店': 127,
    '検証用': 999
}

In [ ]:
# 店舗名（シフトインした店舗）をもとにstore_codeをマッピング
df9['store_code'] = df9['店舗名（シフトインした店舗）'].map(store_dict_report)

if(df9['店舗名（シフトインした店舗）'].count() == df9['store_code'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(df9[df9['store_code'].isnull()]['店舗名（シフトインした店舗）'].unique())

マッピング成功


In [ ]:
df9 = df9[df9['store_code'] != 999]

In [ ]:
df9.shape

(53924, 512)

In [ ]:
# 正規の店舗名を店舗番号からマッピング
df9['store'] = df9['store_code'].map(inverse_store_dict)

df9.drop(columns=['店舗名（シフトインした店舗）'], inplace=True)



In [ ]:
df9.columns

Index(['responseId', 'timestamp', '33044c18', '対象数', 'スタッフ名（フルネーム）', '告知までの課題',
       'SHIRURU在庫状況\nシフトを上がったタイミングで「在庫0かどうか」を選択してください。', 'スタッフナンバー（7桁）',
       'シフトイン日時', 'サービス連携',
       ...
       '28d64205', '60816d0b', '7c1cbc62', '112a9c42', '30cbf9e9', '4e128986',
       '13936530', '1d54b979', 'store_code', 'store'],
      dtype='object', length=512)

In [ ]:
# timestampを基に昇順に並べる
df9.sort_values(by='timestamp', ascending=True, inplace=True)
df9.reset_index(drop=True, inplace=True)

In [ ]:
df9raw = df9.copy()

In [ ]:
df9.rename(columns={
    '対象数': 'target_count',
    'スタッフ名（フルネーム）': 'staff_name',
    'スタッフナンバー（7桁）': 'staff_number',
    'シフトイン日時': 'shift-in_at',
    '非接客時間（OP / CL / 貸切 / 席利用シフト）': 'non-serving_time',
    '接客時間（日中 / PICSシフト）': 'serving_time',
    '接客数': 'customer_count',
    '未告知数': 'non-announcement',
    '誘致数': 'invitation'
    }, inplace=True)

#df9 = df9.drop(columns=['non-serving_time','serving_time',])

df9.columns

Index(['responseId', 'timestamp', '33044c18', 'target_count', 'staff_name',
       '告知までの課題', 'SHIRURU在庫状況\nシフトを上がったタイミングで「在庫0かどうか」を選択してください。',
       'staff_number', 'shift-in_at', 'サービス連携',
       ...
       '28d64205', '60816d0b', '7c1cbc62', '112a9c42', '30cbf9e9', '4e128986',
       '13936530', '1d54b979', 'store_code', 'store'],
      dtype='object', length=512)

In [ ]:
df9['timestamp'] = pd.to_datetime(df9['timestamp'], format='mixed')

# 標準時から日本時間に！
df9['timestamp'] = df9['timestamp'].dt.tz_convert("Asia/Tokyo")

In [ ]:
# 一旦浮動小数点型にしてから整数型に
cols = ['staff_number', 'target_count', 'customer_count', 'non-announcement']
df9[cols] = df9[cols].astype(float).astype(int)

# 数字でないものは0に
df9['invitation'] = df9['invitation'].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

In [ ]:
# shift-in_at は途中まで年がない...
def fix_shift_in_at(shift, timestamp):
    try:
        # すでに年がある場合（YYYY-MM-DD HH:MM）
        if len(shift.split()) == 2 and shift.count('-') == 2:
            return pd.to_datetime(shift, format='%Y-%m-%d %H:%M', errors='coerce')

        # 年がない場合（MM-DD HH:MM）→ timestamp から年を補完し、秒を"00"にする
        year = timestamp.year
        shift_fixed = f"{year}-{shift}:00"  # 秒を追加
        return pd.to_datetime(shift_fixed, format='%Y-%m-%d %H:%M:%S', errors='coerce')

    except Exception as e:
        print(f"エラー: {e} (shift={shift}, timestamp={timestamp})")
        return None

df9['shift-in_at'] = df9.apply(lambda row: fix_shift_in_at(row['shift-in_at'], row['timestamp']), axis=1)

In [ ]:
# 元の行数を記録
original_len = len(df9)

# シフトイン日時とスタッフナンバーが重複している「先のデータ」を削除
df9 = df9[~df9.duplicated(subset=['shift-in_at', 'staff_number'], keep='last')]

# 削除された行数を計算
deleted_rows = original_len - len(df9)

print(f"削除された重複行数: {deleted_rows} 行")

削除された重複行数: 304 行


In [ ]:
df9.shape

(53620, 512)

In [ ]:
df9['timestamp'] = pd.to_datetime(df9['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S'))
df9['shift-in_at'] = pd.to_datetime(df9['shift-in_at'].dt.strftime('%Y-%m-%d %H:%M:%S'))

In [ ]:
# staff_numberとshift-in_atが同じ組み合わせの中で最新（=昇順で最後）のデータを残す
df9 = df9.drop_duplicates(subset=['staff_number', 'shift-in_at'], keep='last')

In [ ]:
# スタッフ名から半角、全角スペースを除く
df9['staff_name'] = df9['staff_name'].str.replace(r'[ 　]+', '', regex=True)

In [ ]:
questions_common = ['responseId', 'timestamp', 'store_code', 'store', 'staff_number',
           'staff_name', 'shift-in_at','customer_count', 'target_count',
           'non-announcement', 'invitation']
bq_report = df9[questions_common].copy()

In [ ]:
df9.shape

(53620, 512)

In [ ]:
bq_report.shape

(53620, 11)

## 店舗別日報シート

In [ ]:
# 質問IDと質問を対応させるマッピングを作成（全店分）
question_mapping = {}
for item in form_stracture:
    question_id = item.get("questionItem", {}).get("question", {}).get("questionId", "No ID")
    question_title = item.get("title", {})
    question_mapping[question_id] = question_title

In [ ]:
dict_jp = {"staff_name": "スタッフ名",
           "staff_number": "スタッフナンバー",
           "shift-in_at": "シフトイン日時",
           "customer_count": "接客数",
           "non-announcement": "未告知数",
           "invitation": "誘致数",
           "store": "店舗",
           "store_code": "店舗番号",
           "target_count": "対象数"}

In [ ]:
stores = ss_report['店舗番号'].unique().tolist()
stores.sort()
print(stores)

[101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 114, 115, 116, 118, 119, 121, 124, 125, 126, 127, 422, 423]


In [ ]:
for store_code in stores:
  questions_store = []

  for item in form_stracture:
    section_code = item.get("section", {})
    if section_code == store_code:
      question_id = item.get("questionItem", {}).get("question", {}).get("questionId", "No ID")
      if question_id in df9.columns:
        questions_store.append(item.get("questionItem", {}).get("question", {}).get("questionId", "No ID"))

  columns_store = questions_common + questions_store
  df_store = df9[columns_store]
  df_store = df_store[df_store['store_code'] == store_code]
  df_store = df_store[df_store['timestamp']>start_date]

  # カラム名を質問内容に変換
  df_store.rename(columns=question_mapping, inplace=True)

  df_store.rename(columns=dict_jp, inplace=True)

  try:
      worksheet = spreadsheet.worksheet(str(store_code))
      worksheet.clear()
  except gspread.exceptions.WorksheetNotFound:
      # シートが存在しない場合は新しくシートを作成
      worksheet = spreadsheet.add_worksheet(title=str(store_code), rows="5000", cols="30")
  set_with_dataframe(worksheet, df_store)
  print(f"\r{store_code}データ書き込み完了！✧٩(ˊωˋ*)و✧", flush=True)

101データ書き込み完了！✧٩(ˊωˋ*)و✧
102データ書き込み完了！✧٩(ˊωˋ*)و✧
103データ書き込み完了！✧٩(ˊωˋ*)و✧
104データ書き込み完了！✧٩(ˊωˋ*)و✧
105データ書き込み完了！✧٩(ˊωˋ*)و✧
106データ書き込み完了！✧٩(ˊωˋ*)و✧
107データ書き込み完了！✧٩(ˊωˋ*)و✧
108データ書き込み完了！✧٩(ˊωˋ*)و✧
110データ書き込み完了！✧٩(ˊωˋ*)و✧
111データ書き込み完了！✧٩(ˊωˋ*)و✧
112データ書き込み完了！✧٩(ˊωˋ*)و✧
114データ書き込み完了！✧٩(ˊωˋ*)و✧
115データ書き込み完了！✧٩(ˊωˋ*)و✧
116データ書き込み完了！✧٩(ˊωˋ*)و✧
118データ書き込み完了！✧٩(ˊωˋ*)و✧
119データ書き込み完了！✧٩(ˊωˋ*)و✧
121データ書き込み完了！✧٩(ˊωˋ*)و✧
124データ書き込み完了！✧٩(ˊωˋ*)و✧
125データ書き込み完了！✧٩(ˊωˋ*)و✧
126データ書き込み完了！✧٩(ˊωˋ*)و✧
127データ書き込み完了！✧٩(ˊωˋ*)و✧
422データ書き込み完了！✧٩(ˊωˋ*)و✧
423データ書き込み完了！✧٩(ˊωˋ*)و✧


## 新フォームデータ抽出

In [ ]:
from googleapiclient.discovery import build
from google.oauth2 import service_account

# フォームの回答とフォームの構造を読み取るアクセス権限
SCOPES = [
    "https://www.googleapis.com/auth/forms.responses.readonly",
    "https://www.googleapis.com/auth/forms.body.readonly"
]

# 秘密鍵が含まれるJSON
SERVICE_ACCOUNT_FILE = f'{path}/daily-report-451612-0a0691c91c6b.json'

# Google APIの認証
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

# APIを操作するオブジェクトの作成(?)
service = build("forms", "v1", credentials=credentials)

In [ ]:
responses = []
page_token = None

# フォームIDを指定（フォームの編集URLで/d/と/editに挟まれる文字列）
form_id = "1-g9wDUw5OJeUTbaMfoFk-XeP56HMQaIGSJ95XVZ6z98"

# 一度に最大5000件しか取得できないので全て取得するまで回す
while True:
    # API リクエスト
    response = service.forms().responses().list(
        formId=form_id,
        pageSize=5000,
        pageToken=page_token
    ).execute()

    # データを追加
    responses.extend(response.get("responses", []))

    # 次のページがあるか確認
    page_token = response.get("nextPageToken")
    if not page_token:
        break  # 次のページがなければ終了

In [ ]:
# 各回答を処理して最初に質問IDを使ってDataFrameを作成
data = []
for res in responses:
    row = {
        "responseId": res["responseId"],
        "timestamp": res["createTime"],
    }
    for question_id, answer in res.get("answers", {}).items():
        # 質問IDごとの回答を追加
        row[question_id] = answer.get("textAnswers", {}).get("answers", [{}])[0].get("value", "")
    data.append(row)

# 最初に質問IDを使ってDataFrameを作成
df_report_raw = pd.DataFrame(data)

df_report_raw.shape

(20, 90)

In [ ]:
# フォームの構造を取得
form = service.forms().get(formId=form_id).execute()

In [ ]:
import re

def add_section_to_items(items):
    current_section = 0

    for item in items:
        # itemが辞書であることを確認
        if not isinstance(item, dict):
            continue

        # 1. ページ区切りの判定と更新
        if 'pageBreakItem' in item:
            title = item.get('title', '')
            # タイトルから数字部分を抽出
            match = re.search(r'(\d+)', title)
            if match:
                current_section = int(match.group(1))
            else:
                current_section = 0

        # 2. 質問アイテムへのsection追加
        elif 'questionItem' in item:
            item['section'] = current_section

    return items

# --- 実行 ---
form_stracture = add_section_to_items(form.get('items', []))

In [ ]:
# 質問IDと質問を対応させるマッピングを作成(全店共通セクションのみ)
question_mapping_common = {}
for item in form_stracture:
  store_code = item.get("section", {})
  if store_code != 0:
    continue
  question_id = item.get("questionItem", {}).get("question", {}).get("questionId", "No ID")
  question_title = item.get("title", {})
  question_mapping_common[question_id] = question_title

In [ ]:
# カラム名を質問内容に変換
df_report_raw.rename(columns=question_mapping_common, inplace=True)

## 新フォームデータ処理

In [ ]:
df9 = df_report_raw.copy()

In [ ]:
# 店舗名（シフトインした店舗）をもとにstore_codeをマッピング
df9['store_code'] = df9['店舗名（シフトインした店舗）'].map(store_dict)

if(df9['店舗名（シフトインした店舗）'].count() == df9['store_code'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(df9[df9['store_code'].isnull()]['店舗名（シフトインした店舗）'].unique())

マッピング成功


In [ ]:
# df9 = df9[df9['store_code'] != 999]

In [ ]:
# timestampを基に昇順に並べる
df9.sort_values(by='timestamp', ascending=True, inplace=True)
df9.reset_index(drop=True, inplace=True)

In [ ]:
df9raw = df9.copy()

In [ ]:
df9.rename(columns={
    '対象数': 'target_count',
    'スタッフ名（フルネーム）': 'staff_name',
    'スタッフナンバー（7桁）': 'staff_number',
    'シフトイン日時': 'shift-in_at',
    '非接客時間（OP / CL / 貸切 / 席利用シフト）': 'non-serving_time',
    '接客時間（日中 / PICSシフト）': 'serving_time',
    '接客数': 'customer_count',
    '未告知数': 'non-announcement',
    '誘致数': 'invitation',
    '店舗名（シフトインした店舗）': 'store'
    }, inplace=True)

In [ ]:
df9['timestamp'] = pd.to_datetime(df9['timestamp'], format='mixed')

# 標準時から日本時間に！
df9['timestamp'] = df9['timestamp'].dt.tz_convert("Asia/Tokyo")

In [ ]:
# 一旦浮動小数点型にしてから整数型に
cols = ['staff_number', 'target_count', 'customer_count', 'non-announcement']
df9[cols] = df9[cols].astype(float).astype(int)

# 数字でないものは0に
df9['invitation'] = df9['invitation'].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

In [ ]:
# shift-in_at は途中まで年がない...
def fix_shift_in_at(shift, timestamp):
    try:
        # すでに年がある場合（YYYY-MM-DD HH:MM）
        if len(shift.split()) == 2 and shift.count('-') == 2:
            return pd.to_datetime(shift, format='%Y-%m-%d %H:%M', errors='coerce')

        # 年がない場合（MM-DD HH:MM）→ timestamp から年を補完し、秒を"00"にする
        year = timestamp.year
        shift_fixed = f"{year}-{shift}:00"  # 秒を追加
        return pd.to_datetime(shift_fixed, format='%Y-%m-%d %H:%M:%S', errors='coerce')

    except Exception as e:
        print(f"エラー: {e} (shift={shift}, timestamp={timestamp})")
        return None

df9['shift-in_at'] = df9.apply(lambda row: fix_shift_in_at(row['shift-in_at'], row['timestamp']), axis=1)

In [ ]:
# 元の行数を記録
original_len = len(df9)

# シフトイン日時とスタッフナンバーが重複している「先のデータ」を削除
df9 = df9[~df9.duplicated(subset=['shift-in_at', 'staff_number'], keep='last')]

# 削除された行数を計算
deleted_rows = original_len - len(df9)

print(f"削除された重複行数: {deleted_rows} 行")

削除された重複行数: 2 行


In [ ]:
df9['timestamp'] = pd.to_datetime(df9['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S'))
df9['shift-in_at'] = pd.to_datetime(df9['shift-in_at'].dt.strftime('%Y-%m-%d %H:%M:%S'))

In [ ]:
# スタッフ名から半角、全角スペースを除く
df9['staff_name'] = df9['staff_name'].str.replace(r'[ 　]+', '', regex=True)

In [ ]:
questions_common = ['responseId', 'timestamp', 'store_code', 'store', 'staff_number',
           'staff_name', 'shift-in_at','customer_count', 'target_count',
           'non-announcement', 'invitation']
bq_report2 = df9[questions_common].copy()

In [ ]:
bq_report2.shape

(18, 11)

# データ集計

## 来店

In [ ]:
df_visiter = df4.copy()

In [ ]:
# 日付のみに変換
df_visiter['date'] = df_visiter['ordered_at'].dt.date

In [ ]:
from os import dup

# dateとstore_codeごとに総来店(visit_total)とDU(visit_du)を集計
df_visiter = df_visiter.groupby(['date', 'store_code'])['conected_id'].agg(
    visit_total='count',  # 行数（全体のカウント）
    visit_du='nunique'      # ユニークな値の個数
)

df_visiter.head()

visit_total  visit_du
date       store_code                       
2023-10-02 101                  21        19
           102                  20        16
           103                   2         1
           104                   4         4
           105                  12        11

In [ ]:
# インデックスをリセット
df_visiter = df_visiter.reset_index()

df_visiter.head()

,date,store_code,visit_total,visit_du
0,2023-10-02,101,21,19
1,2023-10-02,102,20,16
2,2023-10-02,103,2,1
3,2023-10-02,104,4,4
4,2023-10-02,105,12,11


## Meetup

In [ ]:
df8 = df5.copy()

In [ ]:
# 日付のみに変換
df8['date'] = df8['start_at'].dt.date

In [ ]:
# キャンセル有無を1,0に(あり→1,なし→0)
df8['cancell'] = df8['cancell'].notnull().astype(int)

In [ ]:
# date,store_code毎に参加、参加予定、キャンセルのそれぞれの合計
df8 = df8.groupby(['date','store_code'])[['attendance','planned_attendance','cancell']].sum()
df8 = df8.reset_index()

In [ ]:
df8.columns

Index(['date', 'store_code', 'attendance', 'planned_attendance', 'cancell'], dtype='object')

## 外部結合

In [ ]:
# 外部結合
df_daily = pd.merge(df_visiter, df8, on=['date', 'store_code'], how='outer')

# 欠損値を0で埋める
df_daily.fillna(0, inplace=True)

df_daily.shape

(14903, 7)

In [ ]:
# 店舗番号から店舗名
df_daily['store'] = df_daily['store_code'].map(inverse_store_dict)

# カラムの順番
df_daily = df_daily[['date', 'store_code', 'store', 'visit_total', 'visit_du',
                     'attendance', 'planned_attendance', 'cancell']]

In [ ]:
# 店舗番号が有効でないカラムを消す
df_daily = df_daily[(df_daily['store_code'] > 0) & (df_daily['store_code'] < 500)]
df_daily.shape

(14005, 8)

In [ ]:
# 全てのデータが0のレコードを消す
#df_daily = df_daily[~((df_daily['visit_total'] == 0) & (df_daily['attendance'] == 0) & (df_daily['cancell'] == 0))]
df_daily.shape

(14005, 8)

In [ ]:
# 整数型に
int_columns = ['visit_total', 'visit_du', 'attendance', 'planned_attendance', 'cancell']
df_daily[int_columns] = df_daily[int_columns].astype(int)

# 個人データ

In [ ]:
# 来店データから個人データを集計

user_columns_order = ['conected_id','ordered_at', 'store_code','group',  'grade', 'university', 'faculty', 'DU_id']

# DUのみを残す
df4['ordered_at'] = pd.to_datetime(df4['ordered_at'])
df_du = df4.drop_duplicates(subset='DU_id')

# 結合ID毎に集計
df_user_order = df_du[user_columns_order].groupby('conected_id').agg(
    visit=('DU_id', 'nunique'),
    first_ordered_at=('ordered_at', 'min'),
    last_ordered_at=('ordered_at', 'max'),
    store_code=('store_code', 'first'),
    grade=('grade', 'last'),
    university=('university', 'last'),
    faculty=('faculty', 'last')
).reset_index()

df_user_order.shape

(108412, 8)

In [ ]:
df4['ordered_at'] = pd.to_datetime(df4['ordered_at'])

this_year_april = pd.Timestamp(year=pd.Timestamp.today().year, month=4, day=1)

df_visit_since_april = (
    df4[df4['ordered_at'] >= this_year_april]
    .groupby('conected_id')
    .agg(
        visit_since_april=('DU_id', 'nunique')
    )
    .reset_index()
)

df_user_order = df_user_order.merge(
    df_visit_since_april,
    on='conected_id',
    how='left'
)

# 4月以降の来店がない人は 0 にする
df_user_order['visit_since_april'] = df_user_order['visit_since_april'].fillna(0).astype(int)

In [ ]:
df_user_order.head()

,conected_id,visit,first_ordered_at,last_ordered_at,store_code,grade,university,faculty,visit_since_april
0,S109k2,32,2024-04-23 13:41:07,2024-09-30 14:03:18,904,その他,インド工科大学ハイデラバード校,土木工学科,0
1,S10aqo,12,2023-12-12 14:53:30,2024-03-19 15:14:27,904,None,インド工科大学ハイデラバード校,None,0
2,S10rb8,1,2023-10-04 15:03:29,2023-10-04 15:03:29,904,博士1年,インド工科大学ハイデラバード校,電子工学科,0
3,S111111,1,2025-10-01 13:38:00,2025-10-01 13:38:00,111,None,立命館大学(衣笠キャンパス),None,0
4,S116p6,1,2024-08-09 14:21:39,2024-08-09 14:21:39,904,None,インド工科大学ハイデラバード校,None,0


In [ ]:
df_user_meetup = df2.copy()
df_user_meetup['キャンセル'] = df_user_meetup['キャンセル有無'].notnull()
df_user_meetup['キャンセル'].sum()

np.int64(21186)

In [ ]:
# Meetupデータから個人データを集計

user_columns_meetup = ['結合ID','イベント開始日時','会員ID', '大学', '学部', '学年',
                       '参加', 'キャンセル', '店舗番号']



# 結合ID毎に集計
df_user_meetup = df_user_meetup[user_columns_meetup].groupby('結合ID').agg(
    attendance=('参加', 'sum'),
    cancel=('キャンセル', 'sum'),
    first_event_datetime=('イベント開始日時', 'min'),
    last_event_datetime=('イベント開始日時', 'max'),
    store_code=('店舗番号', lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    university=('大学', 'last'),
    faculty=('学部', 'last'),
    grade=('学年', 'last'),
).reset_index()

df_user_meetup.rename(columns={'結合ID': 'conected_id'}, inplace=True)
df_user_meetup.shape

(21740, 9)

In [ ]:
df2['イベント開始日時'] = pd.to_datetime(df2['イベント開始日時'])

this_year_april = pd.Timestamp(year=pd.Timestamp.today().year, month=4, day=1)

df_meetup_since_april = (
    df2[df2['イベント開始日時'] >= this_year_april]
    .groupby('結合ID')
    .agg(
        attendance_since_april=('参加', 'sum')
    )
    .reset_index()
)

df_meetup_since_april.rename(columns={'結合ID': 'conected_id'}, inplace=True)

df_user_meetup = df_user_meetup.merge(
    df_meetup_since_april,
    on='conected_id',
    how='left'
)

# 4月以降の参加がない人は 0 にする
df_user_meetup['attendance_since_april'] = df_user_meetup['attendance_since_april'].fillna(0).astype(int)

In [ ]:
df_user_meetup.head()

,conected_id,attendance,cancel,first_event_datetime,last_event_datetime,store_code,university,faculty,grade,attendance_since_april
0,2025/7/1,0,0,2025-06-27 15:15:00,2025-06-27 15:15:00,102,京都大学,工学部,学部4年,0
1,S14ajx,0,3,2025-06-17 17:30:00,2026-07-29 15:30:00,106,神戸大学,経済学部,学部4年,0
2,S20cf8n,1,0,2026-05-29 13:00:00,2026-05-29 13:00:00,102,京都大学,理学研究科,博士3年,1
3,S29vsls,0,4,2025-04-14 14:00:00,2025-07-02 15:00:00,105,東京大学,経済学部,博士2年,0
4,S29z0lo,7,1,2024-07-31 15:30:00,2024-10-11 14:00:00,112,九州大学,理学部,博士2年,0


In [ ]:
# データフレームを結合IDをキーとして結合
df_user = pd.merge(
    df_user_order,
    df_user_meetup,
    on='conected_id',
    how='outer'
)

df_user.columns

Index(['conected_id', 'visit', 'first_ordered_at', 'last_ordered_at',
       'store_code_x', 'grade_x', 'university_x', 'faculty_x',
       'visit_since_april', 'attendance', 'cancel', 'first_event_datetime',
       'last_event_datetime', 'store_code_y', 'university_y', 'faculty_y',
       'grade_y', 'attendance_since_april'],
      dtype='object')

In [ ]:
# Meetupデータの欠損値を来店データから埋める
df_user['university'] = df_user['university_y'].fillna(df_user['university_x'])
df_user['faculty'] = df_user['faculty_y'].fillna(df_user['faculty_x'])
df_user['grade'] = df_user['grade_y'].fillna(df_user['grade_x'])
df_user['store_code'] = df_user['store_code_y'].fillna(df_user['store_code_x'])

df_user.drop(columns=['university_x', 'faculty_x', 'grade_x', 'store_code_x', 'university_y', 'faculty_y', 'grade_y', 'store_code_y'], inplace=True)

In [ ]:
# 数値の欠損を0にして、数値を整数型に
int_col = ['visit', 'attendance', 'store_code', 'cancel']
df_user[int_col] = df_user[int_col].fillna(0).astype(int)

In [ ]:
df_user['store'] = df_user['store_code'].map(inverse_store_dict)

In [ ]:
bq_user = df_user[df_user['store_code'] < 500]

#MCS

In [ ]:
## スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/11jDJSgYZlVJgMQ5HQy94SuKdWQcv4R5PuMjI0B-dSI4/edit?gid=0#gid=0"
ss_mcs = gc.open_by_url(url)
print(f'{ss_mcs.title}を開きました')

# スプレッドシートを読み込みデータフレーム化
st_mcs = ss_mcs.get_worksheet(0)
df_ss_mcs = get_as_dataframe(st_mcs)
df_ss_mcs.shape

# 数式の結果を取得してDataFrameに変換
df_ss_mcs = get_as_dataframe(st_mcs, evaluate_formulas=True)

# データの確認
print(df_ss_mcs.head())

raw_admin_MCSを開きました
          日付                                 店舗    卒業年度    a    b    c  視聴完了数  \
0  2023/9/18          shirucafe_119tokyo_komaba     NaN  1.0  0.0  1.0    0.0   
1  2023/9/19                          shirucafe  2025.0  1.0  1.0  4.0    3.0   
2  2023/9/19             shirucafe_101doushisha  2029.0  1.0  1.0  3.0    3.0   
3  2023/9/19                shirucafe_108kansai     NaN  0.0  0.0  0.0    0.0   
4  2023/9/19  shirucafe_111ritsumeikan_kinugasa     NaN  1.0  1.0  2.0    1.0   

   視聴数          店舗名   店舗番号  
0  0.0   東京大学(駒場)前店  119.0  
1  NaN          NaN    NaN  
2  3.0      同志社大学前店  101.0  
3  0.0       関西大学前店  108.0  
4  1.0  立命館大学(衣笠)前店  111.0  


In [ ]:
df_ss_mcs.columns

Index(['日付', '店舗', '卒業年度', 'a', 'b', 'c', '視聴完了数', '視聴数', '店舗名', '店舗番号'], dtype='object')

In [ ]:
#バックアップをとる
dfm = df_ss_mcs.copy()

In [ ]:
# 店舗名をもとに店舗番号をマッピング
dfm['店舗番号'] = dfm['店舗名'].map(store_dict)

if(dfm['店舗名'].count() == dfm['店舗番号'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(dfm[dfm['店舗番号'].isnull()]['店舗名'].unique())

マッピング成功


In [ ]:
# カラムを消す
dfm.drop(columns=['a','b','c','卒業年度'], inplace = True)

# 日付型に
dfm['日付'] = pd.to_datetime(dfm['日付']).dt.date

#整数型に
#dfm[['店舗番号']] = dfm[['店舗番号']].astype(int)

# NaN を 0 に変えて int に変換
dfm['視聴数'] = dfm['視聴数'].fillna(0).astype(int)
dfm['視聴完了数'] = dfm['視聴完了数'].fillna(0).astype(int)
dfm['店舗番号'] = dfm['店舗番号'].fillna(0).astype(int)

dfm.columns

Index(['日付', '店舗', '視聴完了数', '視聴数', '店舗名', '店舗番号'], dtype='object')

In [ ]:
column_mapping = {
    '日付':'date',
    '店舗':'store_id',
    '視聴完了数':'comp_viewing',
    '視聴完了数_修正':'comp_viewing_fix',
    '視聴数':'viewing',
    '店舗名':'store',
    '店舗番号':'store_num'
}

# カラム名を英語に変換
dfm.rename(columns=column_mapping, inplace=True)

In [ ]:
bq_mcs = dfm.copy()

#SHIRURU

In [ ]:
## スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/1IfieJcqJPft1dFfsGGO3n2buPxL_TE07O-ZZyy4hqLI/edit?gid=0#gid=0"
ss_srr = gc.open_by_url(url)
print(f'{ss_srr.title}を開きました')

# スプレッドシートを読み込みデータフレーム化
st_srr = ss_srr.get_worksheet(0)
df_ss_srr = get_as_dataframe(st_srr)
df_ss_srr.shape

# 数式の結果を取得してDataFrameに変換
df_ss_srr = get_as_dataframe(st_srr, evaluate_formulas=True)

# データの確認
print(df_ss_srr.head())

raw_admin_SHIRURU_newを開きました
       会員ID              企業名  \
0  363055.0        三菱重工業株式会社   
1  383557.0  株式会社ニトリホールディングス   
2  384643.0          株式会社ロッテ   
3  363055.0          株式会社ロッテ   
4  363055.0      株式会社構造計画研究所   

                                                 店舗      大学    学部    学年  性別  \
0  BiZCAFE(関西学院大学神戸三田)店 (Kwansei Gakuin University)  関西学院大学  理工学部  学部4年  男性   
1              関西学院大学前店 (Kwansei Gakuin University)  関西学院大学  教育学部  学部4年  女性   
2                        慶應義塾大学前店 (Keio University)  慶應義塾大学  経済学部  学部4年  男性   
3  BiZCAFE(関西学院大学神戸三田)店 (Kwansei Gakuin University)  関西学院大学  理工学部  学部4年  男性   
4  BiZCAFE(関西学院大学神戸三田)店 (Kwansei Gakuin University)  関西学院大学  理工学部  学部4年  男性   

              アクション形態                   日時     卒業年月 文理区分     修正_日時   店舗番号  \
0  知るカフェ/BiZCAFEで話したい   2023-10-03 7:36:11  2025年4月   理系  202310.0  422.0   
1  知るカフェ/BiZCAFEで話したい  2023-10-03 22:25:49  2025年3月   文系  202310.0  118.0   
2  知るカフェ/BiZCAFEで話したい   2023-10-04 3:31:52  2025年3月   文系  202310.0  107.0

In [ ]:
df_ss_srr.columns

Index(['会員ID', '企業名', '店舗', '大学', '学部', '学年', '性別', 'アクション形態', '日時', '卒業年月',
       '文理区分', '修正_日時', '店舗番号', '店舗名', 'SCREEN条件', 'KOKUSAI条件', '配布判定', '配布単価',
       'Unnamed: 21'],
      dtype='object')

In [ ]:
#バックアップをとる
dfsr = df_ss_srr.copy()

In [ ]:
# 店舗名をもとに店舗番号をマッピング
dfsr['店舗番号'] = dfsr['店舗名'].map(store_dict)

if(dfsr['店舗名'].count() == dfsr['店舗番号'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(dfsr[dfsr['店舗番号'].isnull()]['店舗名'].unique())

マッピング成功


In [ ]:
# カラムを消す
dfsr.drop(columns=['Unnamed: 21','SCREEN条件', 'KOKUSAI条件','アクション形態','卒業年月'], inplace = True)

# 日時型に
dfsr['日時'] = pd.to_datetime(dfsr['日時'], errors='coerce')
dfsr['修正_日時'] = pd.to_datetime(dfsr['修正_日時']).dt.date

dfsr['会員ID'] = dfsr['会員ID'].fillna(0).astype(int)
dfsr['配布判定'] = dfsr['配布判定'].fillna(0).astype(int)

#整数型に
dfsr[['会員ID']] = dfsr[['会員ID']].astype(int)
dfsr[['配布判定']] = dfsr[['配布判定']].astype(int)

# NaN を 0 に変えて int に変換
dfsr['店舗番号'] = dfsr['店舗番号'].fillna(0).astype(int)


dfsr.columns

Index(['会員ID', '企業名', '店舗', '大学', '学部', '学年', '性別', '日時', '文理区分', '修正_日時',
       '店舗番号', '店舗名', '配布判定', '配布単価'],
      dtype='object')

In [ ]:
column_mapping = {
    '会員ID':'guest_num',
    '企業名':'company',
    '店舗':'store_id',
    '大学':'university',
    '学部':'faculty',
    '学年':'grade',
    '性別':'gender',
    '日時':'date',
    '文理区分':'bunri',
    '修正_日時':'date_fix',
    '店舗番号':'store_num',
    '店舗名':'store',
    '配布判定':'handing',
    '配布単価':'cost'
}

# カラム名を英語に変換
dfsr.rename(columns=column_mapping, inplace=True)

In [ ]:
bq_srr = dfsr.copy()

# 目標値

In [ ]:
## スプレッドシートを開く
url = "https://docs.google.com/spreadsheets/d/1P0cnzgYwEe1wYYeTuhi0qiv7c81q8cCy4T4JfnJ3f-Q/edit?gid=769489446#gid=769489446"
ss_goal = gc.open_by_url(url)
print(f'{ss_goal.title}を開きました')

# スプレッドシートを読み込みデータフレーム化
st_goal = ss_goal.get_worksheet(1)
df_ss_goal = get_as_dataframe(st_goal)
df_ss_goal.shape

# 数式の結果を取得してDataFrameに変換
df_ss_goal = get_as_dataframe(st_goal, evaluate_formulas=True)

# スプレッドシートを読み込みデータフレーム化
st_goal2 = ss_goal.get_worksheet(2)
df_ss_goal2 = get_as_dataframe(st_goal2)
df_ss_goal2.shape

# 数式の結果を取得してDataFrameに変換
df_ss_goal2 = get_as_dataframe(st_goal2, evaluate_formulas=True)

# データの確認
print(df_ss_goal.head())

raw_目標管理シートを開きました
       店舗名   店舗番号      目標月          結合キー      結合ナンバー         目標日   休日   曜日  \
0  同志社大学前店  101.0  2025/10  同志社大学前店45931  45931101.0  2025/10/01  NaN  水曜日   
1  同志社大学前店  101.0  2025/10  同志社大学前店45931  45931101.0  2025/10/02  NaN  木曜日   
2  同志社大学前店  101.0  2025/10  同志社大学前店45931  45931101.0  2025/10/03  NaN  金曜日   
3  同志社大学前店  101.0  2025/10  同志社大学前店45931  45931101.0  2025/10/04   休日  土曜日   
4  同志社大学前店  101.0  2025/10  同志社大学前店45931  45931101.0  2025/10/05   休日  日曜日   

    祝日  総来店目標     DU来店目標  Meetup参加目標  SHIRURU目標  MCS目標  
0  NaN    0.0  77.181818    3.954545        0.0   15.5  
1  NaN    0.0  77.181818    3.954545        0.0   15.5  
2  NaN    0.0  77.181818    3.954545        0.0   15.5  
3  NaN    0.0   0.000000    0.000000        0.0    0.0  
4  NaN    0.0   0.000000    0.000000        0.0    0.0  


In [ ]:
# 外部結合
df_ss_goal1 = pd.concat([df_ss_goal, df_ss_goal2], join='outer', ignore_index=True)
df_ss_goal1.shape

(29288, 14)

In [ ]:
# 重複データ削除
len_before = len(df_ss_goal1)
df_ss_goal1.drop_duplicates(inplace=True)
len_after = len(df_ss_goal1)
print(f'{len_before - len_after}行の重複データが削除されました')

14139行の重複データが削除されました


In [ ]:
df_ss_goal1.columns

Index(['店舗名', '店舗番号', '目標月', '結合キー', '結合ナンバー', '目標日', '休日', '曜日', '祝日',
       '総来店目標', 'DU来店目標', 'Meetup参加目標', 'SHIRURU目標', 'MCS目標'],
      dtype='object')

In [ ]:
#バックアップをとる
dfg = df_ss_goal1.copy()

In [ ]:
# 店舗名をもとに店舗番号をマッピング
dfg['店舗番号'] = dfg['店舗名'].map(store_dict)

if(dfg['店舗名'].count() == dfg['店舗番号'].count()):
  print("マッピング成功")
else:
  print("マッピングに漏れあり")
  print(dfg[dfg['店舗番号'].isnull()]['店舗名'].unique())

マッピング成功


In [ ]:
# カラムを消す
dfg.drop(columns=['祝日','休日'], inplace = True)

# 日付型に
dfg['目標日'] = pd.to_datetime(dfg['目標日']).dt.date

# 年月だけを表す形式に
dfg['目標月'] = pd.to_datetime(dfg['目標月'], format='%Y/%m')

dfg[['店舗番号','総来店目標']] = dfg[['店舗番号','総来店目標']].fillna(0).astype(int)

#整数型に
dfg[['店舗番号','総来店目標']] = dfg[['店舗番号','総来店目標']].astype(int)


#小数型に
dfg[['DU来店目標','Meetup参加目標','SHIRURU目標','MCS目標']] = dfg[['DU来店目標','Meetup参加目標','SHIRURU目標','MCS目標']].astype(float)
dfg.columns

Index(['店舗名', '店舗番号', '目標月', '結合キー', '結合ナンバー', '目標日', '曜日', '総来店目標', 'DU来店目標',
       'Meetup参加目標', 'SHIRURU目標', 'MCS目標'],
      dtype='object')

In [ ]:
column_mapping = {
    '店舗名':'store',
    '店舗番号':'store_code',
    '目標月':'target_month',
    '結合キー':'connection_key',
    '結合ナンバー':'connection_number',
    '目標日':'target_day',
    '曜日':'day',
    '総来店目標':'total_goal',
    'DU来店目標':'DU_goal',
    'Meetup参加目標':'Meetup_goal',
    'SHIRURU目標':'SHIRURU_goal',
    'MCS目標':'MCS_goal'
}

# カラム名を英語に変換
dfg.rename(columns=column_mapping, inplace=True)

In [ ]:
bq_goal = dfg.copy()

## 月毎目標値

In [ ]:
st_goal = ss_goal.get_worksheet(0)
df_goal = get_as_dataframe(st_goal)
df_goal.shape

(470, 15)

In [ ]:
df_goal.columns

Index(['目標月', '店舗番号', 'store',
       '={"結合ナンバー" ; arrayformula(IF(A2:A1009="",,to_text(A2:A1009*1000+B2:B1009)))}',
       '総来店目標_monthly', 'DU来店目標_monthly', 'Meetup目標_monthly',
       'SHIRURU目標_monthly', 'MCS目標_monthly', '総来店目標_daily', 'DU来店目標_daily',
       'Meetup目標_daily', 'SHIRURU目標_daily', 'MCS目標_daily', 'DU新来店目標_monthly'],
      dtype='object')

In [ ]:
df_goal = df_goal[['目標月', '店舗番号', 'store',
       '総来店目標_monthly', 'DU来店目標_monthly', 'Meetup目標_monthly',
       'SHIRURU目標_monthly', 'MCS目標_monthly']]

In [ ]:
df_goal['目標月'] = pd.to_datetime(df_goal['目標月'], format='mixed', errors='coerce')
df_goal['店舗番号'] = df_goal['店舗番号'].fillna(0).astype(int)

In [ ]:
column_mapping = {
    '目標月':'target_month',
    '店舗番号':'store_code',
    '総来店目標_monthly':'kpi_order',
    'DU来店目標_monthly':'kpi_du',
    'Meetup目標_monthly':'kpi_meetup',
    'SHIRURU目標_monthly':'kpi_shiruru',
    'MCS目標_monthly':'kpi_mcs'
}

# カラム名を英語に変換
df_goal.rename(columns=column_mapping, inplace=True)

In [ ]:
df_goal.dropna(subset=['target_month'], inplace=True)

In [ ]:
df_goal.columns

Index(['target_month', 'store_code', 'store', 'kpi_order', 'kpi_du',
       'kpi_meetup', 'kpi_shiruru', 'kpi_mcs'],
      dtype='object')

In [ ]:
bq_goal_monthly = df_goal.copy()

# BigQuery

In [ ]:
from google.cloud import bigquery

# BigQueryクライアントの作成
client = bigquery.Client()

# プロジェクトID
project_id = 'fair-solution-453613-e2'
data_set = '202506'

In [ ]:
import json

type_dict = {
    "object": "STRING",
    "int64": "INTEGER",
    "float64": "FLOAT",
    "datetime64[ns]": "DATETIME",
    "bool": "BOOLEAN"
}

# スキーマ設定の大体はやってくれる関数
def schema_maker(bq_order):
    """DataFrameの全カラムのnameとtypeをJSON形式で出力"""
    columns_info = [{"name": col, "type": type_dict.get(str(dtype), "STRING")} for col, dtype in bq_order.dtypes.items()]
    return json.dumps(columns_info, indent=4)

# JSON出力
#json_output = schema_maker(bq_meetup)
#print(json_output)

In [ ]:
table_name = '来店'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_order.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/4205064262.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_order.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 9020.01it/s]


In [ ]:
table_name = 'Meetup参加'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_meetup.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/1388651494.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_meetup.to_gbq(table_id, project_id=project_id, if_exists="replace")
/usr/local/lib/python3.12/dist-packages/pandas_gbq/schema/pandas_to_bigquery.py:159: UserWarning: Could not determine the type of columns: bukatsu_id
  warnings.warn(msg)
100%|██████████| 1/1 [00:00<00:00, 10951.19it/s]


In [ ]:
table_name = 'イベント'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_event.to_gbq(table_id, project_id=project_id, if_exists="replace")

NameError: name 'bq_event' is not defined

In [ ]:
bq_report2.shape

(18, 11)

In [ ]:
table_name = '日報'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_report.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_1582/1926415296.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_report.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 7358.43it/s]


In [ ]:
table_name = '日報_new'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_report2.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_1582/3351371818.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_report2.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 10155.70it/s]


In [ ]:
table_name = '日次データ'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
df_daily.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/3482683032.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_daily.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 10255.02it/s]


In [ ]:
table_name = 'ユーザー'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_user.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/3841371321.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_user.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 11881.88it/s]


In [ ]:
table_name = '目標値'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_goal.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/1005736412.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_goal.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 12905.55it/s]


In [ ]:
table_name = '月毎目標値'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_goal_monthly.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/44721861.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_goal_monthly.to_gbq(table_id, project_id=project_id, if_exists="replace")
/usr/local/lib/python3.12/dist-packages/pandas_gbq/schema/pandas_to_bigquery.py:159: UserWarning: Could not determine the type of columns: kpi_order
  warnings.warn(msg)
100%|██████████| 1/1 [00:00<00:00, 11037.64it/s]


In [ ]:
table_name = 'MCS'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_mcs.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/63460444.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_mcs.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 12595.51it/s]


In [ ]:
table_name = 'SHIRURU'

# BigQueryのテーブルに書き込み
table_id = f'{project_id}.{data_set}.{table_name}'
bq_srr.to_gbq(table_id, project_id=project_id, if_exists="replace")

/tmp/ipykernel_4062/3248572641.py:5: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  bq_srr.to_gbq(table_id, project_id=project_id, if_exists="replace")
100%|██████████| 1/1 [00:00<00:00, 10512.04it/s]


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# 現在の日時を取得
now = datetime.now(ZoneInfo("Asia/Tokyo"))

print(f"最終更新日時: {now}")

最終更新日時: 2026-06-30 15:21:49.913999+09:00


In [ ]:
last_online = bq_meetup[(bq_meetup['attendance'] == 1)&(bq_meetup['meetup_type'] == "オンライン")]['start_at'].max()
last_taimen = bq_meetup[(bq_meetup['attendance'] == 1)&(bq_meetup['meetup_type'] == "対面")]['start_at'].max()
last_order = bq_order['ordered_at'].max()
last_mcs = bq_mcs[bq_mcs['viewing']>0]['date'].max()

print("各項目最終日時")
print(f'オンライン参加: {last_online}')
print(f'対面参加: {last_taimen}')
print(f'注文: {last_order}')
print(f'MCS: {last_mcs}')

各項目最終日時
オンライン参加: 2026-06-29 14:00:00
対面参加: 2026-06-29 17:00:00
注文: 2026-06-29 20:20:00
MCS: 2026-06-29
